# CI/CD for a Genie space, end to end
### The simple Genie-as-Code bundle for FreshCart: what every file does, how a change travels from the dev UI to prod, and how it is protected on the way

This notebook explains the CI/CD process of [`genie-as-code-review/`](../genie-as-code-review/), the **simple**
way to manage the FreshCart Genie space (*FreshCart Sales & Stock Assistant*) as code: **one JSON file, one bundle
resource, one GitHub Actions workflow**. It is written for people who know Genie but are new to CI/CD.

By the end you will be able to answer:

- What exactly is deployed, where, by whom and when?
- How does a change made in the Genie UI get into git, and from git into QA and prod?
- What does each quality gate check, what does it *not* check, and what happens when one fails?
- How do you roll back, and what happens if someone edits prod by hand?
- What has to be set up on Databricks and GitHub before the first real deployment?

**Everything in Part 5 really runs:** the real Databricks CLI deploys the real bundle, and the bundle's own workflow
file (`deploy.yml`) is executed step by step, against three local **stand-in workspaces** (dev, qa, prod). You need no
Databricks account to follow along.

## Contents
| Part | | |
|---|---|---|
| 1 | CI/CD in one page | the ideas, the vocabulary, the rules of the game |
| 2 | The process in one picture | environments, who deploys what, when |
| 3 | The building blocks, file by file | bundle config, resource, space JSON, scripts, tests, workflow |
| 4 | One definition, three environments | what each target resolves to, and why the catalog script exists |
| 5 | **Hands-on: the full lifecycle** | first release, authoring loop, a bad change caught, rollback, drift, a broken pull request |
| 6 | Going live | one-time setup on Databricks and GitHub |
| 7 | Runbook | what to do when something fails |
| 8 | Simple bundle or full bundle? | when to move to `genie_bundle/` |
| 9 | Summary and glossary | |

**To run it yourself** (Linux, macOS or WSL; the workflow steps are bash):
1. Install the Databricks CLI **1.14 or newer**:
   `curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh`
2. `pip install -r notebooks/requirements.txt -r genie_bundle/requirements-dev.txt` (and have `git` and `jq`)
3. *Run All* (about 5 minutes; the first run also builds the local FreshCart warehouse).

---
# 1 · CI/CD in one page

### What is being delivered?
A **Genie space** is not code: it is *configuration*. It says which tables and metric views Genie may use, how the
business talks (instructions), which questions have a trusted answer (example SQL), which functions are trusted, and
which questions it must get right (benchmarks). Databricks can export all of it as one JSON document, the
**serialized space**. That JSON is what we version, review, test and deploy.

### CI and CD
| Term | Meaning here |
|---|---|
| **Continuous Integration (CI)** | every proposed change (a *pull request*) is checked automatically before it may be merged into `main` |
| **Continuous Delivery (CD)** | every merged change is deployed automatically to dev and QA and tested there; prod is one approved click away |

### Why bother, for a configuration?
Without CI/CD, a Genie space is edited by hand in each workspace. Nobody can say what changed, when, or why; dev, QA
and prod drift apart; a well-meant edit can break answers in prod and there is no "undo". With CI/CD:

| Need | How it is met |
|---|---|
| **History** | every change is a git commit: who, when, what, why |
| **Review** | changes go through pull requests; the diff of the JSON shows exactly what changes |
| **Same thing everywhere** | QA and prod receive the *same* JSON that was tested |
| **Quality gates** | automatic checks stop a broken space before users see it |
| **Undo** | rolling back = redeploying an earlier commit |
| **Separation of duties** | people author in dev; only the pipeline (a service principal) deploys to QA and prod, prod after an approval |

### The rules of the game
1. **Git is the source of truth.** What is on `main` is what should be running.
2. **You author in dev**, in the Genie UI, where you can try things. The UI is a great editor, but its edits are not
   versioned until you export them into git (`bundle generate`).
3. **Only the pipeline deploys to QA and prod.** Nobody edits those spaces by hand.
4. **Every deployment goes through the same gates**, in the same order: validate → dev + smoke test → QA + benchmark
   gate → approval → prod + smoke test.

### The tools
| Tool | Role |
|---|---|
| **Databricks Asset Bundle** (DAB, also called *Declarative Automation Bundle*) | describes the Genie space in YAML/JSON and deploys it to a **target** with `databricks bundle deploy -t <target>` |
| **Databricks CLI** | runs the bundle commands: `validate`, `deploy`, `plan`, `summary`, `generate`, `deployment bind` |
| **GitHub** | stores the repository, runs pull requests and reviews |
| **GitHub Actions** | runs the workflow (`deploy.yml`) on every pull request, push and manual run |
| **Databricks SDK for Python** | used by the two test scripts to talk to Genie (conversations, benchmark runs) |

---
# 2 · The process in one picture

```
 AUTHOR (dev workspace)            GIT (GitHub)                          GITHUB ACTIONS  (.github/workflows/deploy.yml)
 ──────────────────────            ────────────                          ─────────────────────────────────────────────

 1 Dana improves the space         3 commit on a branch,
   in the Genie UI                   open a pull request ─────────────► PR:    validate  (JSON + bundle validate)
          │                                 │                                     │ must pass before merging
 2 databricks bundle generate               │ review + merge
   (UI → src/*.geniespace.json) ────────────┘                                     ▼
                                   4 main  ──────────────────────────► push:  validate ─► dev  ───────► QA ────────────┐
                                                                                         deploy        deploy           │
                                                                                         + smoke test  + benchmark gate │
                                                                                                                        │
                                   5 manual run, target = prod ──────► dispatch: validate ─► dev ─► QA (gate) ─► ⏸ approval
                                                                                                                  │
                                                                                                                  ▼
                                                                                                    prod deploy + smoke test
```

### Environments
| Target | Bundle mode | Workspace | Data (catalog) | Space title | Deployed by | When |
|---|---|---|---|---|---|---|
| `dev` | development | dev | `freshcart_dev` | `[dev <deployer>] FreshCart Sales & Stock Assistant` | each developer (own copy) and CI | by hand, and on every push to `main` |
| `qa` | production | qa | `freshcart_qa` | `FreshCart Sales & Stock Assistant [QA]` | CI, as the QA service principal | every push to `main` |
| `prod` | production | prod | `freshcart` | `FreshCart Sales & Stock Assistant` | CI, as the prod service principal, after approval | manual run with target `prod` |

**Development mode** gives every deployer their own copy (the `[dev <name>]` prefix), so developers never overwrite
each other. **Production mode** deploys exactly one copy, to a fixed place, and refuses risky shortcuts.

The **data** comes from the FreshCart data bundle in [`../databricks`](../databricks/): it builds the bronze, silver,
gold and semantic layers in each environment's catalog. This notebook is only about the Genie space on top of it.

---
# 3 · The building blocks, file by file

```
genie-as-code-review/                          (the root of its own repository in real life)
├── databricks.yml                             the bundle: name, engine, variables, targets dev / qa / prod
├── resources/
│   └── freshcart_assistant.genie_space.yml    the Genie space resource: title, warehouse, permissions, file_path
├── src/
│   └── freshcart_assistant.geniespace.json    the space itself (serialized space), exported by `bundle generate`
├── scripts/
│   └── set_catalog.sh                         points the JSON at the qa / prod catalog before deploying there
├── tests/
│   ├── test_genie.py                          smoke test: can the deployed space answer?
│   └── benchmark_gate.py                      benchmark gate: does it answer correctly?
└── .github/workflows/
    └── deploy.yml                             the CI/CD workflow
```

First, the setup for this notebook: find the repository and check the Databricks CLI.

In [1]:
import os, sys, json, re, shutil, subprocess, tempfile, textwrap
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd()
while not (ROOT / "genie-as-code-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT / "tests"), str(ROOT / "genie_bundle" / "scripts"), str(ROOT)]
SIMPLE = ROOT / "genie-as-code-review"

import pandas as pd
import yaml
pd.set_option("display.max_colwidth", 140)

CLI = os.environ.get("DATABRICKS_CLI") or shutil.which("databricks")
version = subprocess.run([CLI, "--version"], capture_output=True, text=True).stdout.strip() if CLI else None
assert version and tuple(map(int, re.search(r"v(\d+)\.(\d+)", version).groups())) >= (1, 14), \
    "Install the Databricks CLI 1.14+: curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh"
assert shutil.which("git") and shutil.which("jq") and shutil.which("bash"), "git, jq and bash are needed for Part 5"
print(version)

def show(relative_path):
    """Print a file of the bundle."""
    print((SIMPLE / relative_path).read_text())

Databricks CLI v1.18.0


### 3.1 · `databricks.yml`: the bundle

In [2]:
show("databricks.yml")

# =============================================================================
# Genie-as-Code (simple version) — FreshCart Sales & Stock Assistant
# Purpose:  Manage the FreshCart Genie space as a versioned, promotable bundle resource
#           with as few moving parts as possible: one JSON file, one resource, one workflow.
#           Author in dev -> curate in UI -> `bundle generate` back to source ->
#           test -> promote to QA -> Prod. See README.md for the full lifecycle.
#           The full-featured version of the same space is ../genie_bundle.
#
# NOTE: Genie spaces need the `direct` deploy engine (Terraform engine is NOT supported).
#
# Authentication is not configured here: locally use `-p <profile>` (or DATABRICKS_CONFIG_PROFILE),
# in CI the DATABRICKS_HOST / DATABRICKS_CLIENT_ID / DATABRICKS_CLIENT_SECRET environment variables.
# =============================================================================

bundle:
  name: freshcart-genie-simple
  # Genie spaces 

What to notice:

| Line(s) | Meaning |
|---|---|
| `bundle.name` | the bundle's identity; it is part of the folder the bundle deploys to (`.bundle/freshcart-genie-simple/<target>`) |
| `engine: direct` | Genie spaces can only be deployed by the CLI's *direct* engine (not the older Terraform engine) |
| `databricks_cli_version` | the CLI refuses to run with an older version, so nobody deploys with a CLI that doesn't know `genie_spaces` |
| `include` | every YAML file in `resources/` is part of the bundle |
| `variables` | values that differ per environment or per company: the **catalog**, the **warehouse** (looked up **by name** in each workspace, so you never hard-code an id), a **title suffix**, the **business users group** |
| `targets` | the three environments. Each sets its `mode` and its variable values; `qa` and `prod` set a fixed `root_path`, so exactly one copy exists whoever deploys |
| no `host`, no `profile` | the workspace and the credentials come from the environment: in CI from `DATABRICKS_HOST` / `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET`, on a laptop from `-p <profile>`. (Pin `host:` once you know your URLs: the CLI then refuses to deploy a target to the wrong workspace.) |

### 3.2 · The Genie space resource

In [3]:
show("resources/freshcart_assistant.genie_space.yml")

# Genie space resource — the single starring artifact of this bundle.
# The serialized definition (tables, instructions, sample questions, example SQL, benchmarks)
# lives in src/freshcart_assistant.geniespace.json. Curate in the UI, then run
#   databricks bundle generate genie-space --resource freshcart_assistant --force
# to capture those edits back into that JSON before promoting.
resources:
  genie_spaces:
    freshcart_assistant:
      title: "FreshCart Sales & Stock Assistant${var.title_suffix}"
      description: >-
        Ask about sales, margin, baskets, promotions and stock across FreshCart stores and online.
        Figures are USD, exclude tax and use the FreshCart fiscal calendar.
      file_path: ../src/freshcart_assistant.geniespace.json
      warehouse_id: ${var.warehouse_id}
      # Without this only the deployer (the CI service principal in qa/prod) could open the space.
      # Business users also need SELECT on the FreshCart tables: Genie runs queries as the perso

The resource holds the settings that are **not** part of the space's content: its title, description, SQL warehouse and
**permissions**. The content comes from `file_path`, the JSON file. `${var.title_suffix}` makes the QA copy read
"... [QA]" so nobody mistakes it for prod.

**Permissions matter.** In QA and prod the space is created by the pipeline's service principal, so without this block
only that service principal (and admins) could open it. `CAN_RUN` lets business users ask questions; they also need
`SELECT` on the FreshCart tables, because Genie runs every query **as the person asking** (row-level security keeps
working). The data bundle grants that in `governance/01_security.sql`.

### 3.3 · The space itself: `src/freshcart_assistant.geniespace.json`

This is the **serialized space**: exactly what Databricks exports with `bundle generate genie-space`. Its sections:

In [4]:
import space_tools as T                         # helpers shared with the full bundle (genie_bundle/scripts)
space = json.loads((SIMPLE / "src/freshcart_assistant.geniespace.json").read_text())

purpose = {
    "data_sources.metric_views":         "governed metrics (net sales, margin, stock...) Genie should prefer",
    "data_sources.tables":               "tables Genie may query, with per-column settings (entity matching...)",
    "instructions.text_instructions":    "the business context: fiscal calendar, currency, when to ask a clarification",
    "instructions.example_question_sqls":"trusted example queries: 'for this question, this SQL is right'",
    "instructions.sql_functions":        "trusted SQL functions (like-for-like growth)",
    "config.sample_questions":           "the questions shown to users as suggestions",
    "benchmarks.questions":              "the exam: questions with the expected SQL, run by the QA gate",
}
rows = []
for name, why in purpose.items():
    a, b = name.split(".")
    rows.append({"section": name, "items": len(space.get(a, {}).get(b, [])), "what it is for": why})
display(pd.DataFrame(rows))
print("format version:", space["version"])

,section,items,what it is for
0,data_sources.metric_views,2,"governed metrics (net sales, margin, stock...) Genie should prefer"
1,data_sources.tables,2,"tables Genie may query, with per-column settings (entity matching...)"
2,instructions.text_instructions,1,"the business context: fiscal calendar, currency, when to ask a clarification"
3,instructions.example_question_sqls,4,"trusted example queries: 'for this question, this SQL is right'"
4,instructions.sql_functions,1,trusted SQL functions (like-for-like growth)
5,config.sample_questions,4,the questions shown to users as suggestions
6,benchmarks.questions,8,"the exam: questions with the expected SQL, run by the QA gate"


format version: 2


One trusted example and the benchmark for the same question. The **example** teaches Genie; the **benchmark** checks
that Genie's answer returns the same result as the expected SQL:

In [5]:
q = "What were net sales and margin by region last week?"
example = next(e for e in space["instructions"]["example_question_sqls"] if "".join(e["question"]) == q)
bench = next(b for b in space["benchmarks"]["questions"] if "".join(b["question"]) == q)
print("EXAMPLE  id", example["id"]);  print("".join(example["sql"]), "\n")
print("BENCHMARK id", bench["id"]);   print("".join(bench["answer"][0]["content"]))

EXAMPLE  id 08131930aafcd3a8773021c860fd5d37
SELECT region,
       MEASURE(net_sales)        AS net_sales,
       MEASURE(gross_margin)     AS gross_margin,
       MEASURE(gross_margin_pct) AS gross_margin_pct
FROM   freshcart_dev.semantic.sales_metrics
WHERE  is_last_completed_fiscal_week
GROUP BY ALL
ORDER BY net_sales DESC 

BENCHMARK id 9077e4008a11a00d000741a3b4cb64ff
SELECT region,
       MEASURE(net_sales)        AS net_sales,
       MEASURE(gross_margin)     AS gross_margin,
       MEASURE(gross_margin_pct) AS gross_margin_pct
FROM   freshcart_dev.semantic.sales_metrics
WHERE  is_last_completed_fiscal_week
GROUP BY ALL
ORDER BY net_sales DESC


Three details of the format that the pipeline relies on:

1. **The catalog is `freshcart_dev` everywhere** in the file, because the file is exported from the dev space. Part 4
   explains how QA and prod get their own catalog.
2. **Ids are 32 lowercase hex characters and lists are sorted** (by id, identifier or column name). Databricks writes
   it that way; hand edits must keep it so.
3. **The file is pretty-printed with sorted keys**, exactly as `bundle generate` writes it, so exporting the UI again
   produces a diff of only what changed.

What the FreshCart answer to that question looks like, computed on the local copy of the FreshCart data (the same
SQL Databricks would run):

In [6]:
from freshcart import benchmarks, config as C, pipeline
from freshcart.db import connect
if not (C.WAREHOUSE_DIR / "gold.db").exists():
    pipeline.run(full_refresh=True, verbose=False)          # builds the local FreshCart warehouse (about 20 s)
cols, rows = benchmarks.run_sql(connect(), "".join(example["sql"]))
pd.DataFrame(rows, columns=cols)

,region,net_sales,gross_margin,gross_margin_pct
0,Northeast,5579.91,2003.78,0.359106
1,Canada,2706.50,1020.47,0.377044
2,Midwest,2543.06,911.63,0.358478
3,Southeast,1065.82,388.83,0.364818
4,West,939.99,328.42,0.349387


### 3.4 · The static checks (no workspace needed)
Before anything is deployed, the repository's tests (`tests/test_simple_genie_bundle.py`) check the JSON the way a
reviewer would want: valid format and ids, every referenced object declared as a data source, and **every example
and benchmark SQL actually runs** on the FreshCart data:

In [7]:
errors, warnings = T.validate(T.neutralise(space, "freshcart_dev"))
print("format errors:", errors or "none", "| warnings:", warnings or "none")
print("SQL that fails on the FreshCart data:", T.check_sql_locally(space) or "none")
print("written exactly as bundle generate writes it:",
      (SIMPLE / "src/freshcart_assistant.geniespace.json").read_text() == json.dumps(space, indent=2, sort_keys=True, ensure_ascii=False) + "\n")

format errors: none | warnings: none


SQL that fails on the FreshCart data: none
written exactly as bundle generate writes it: True


### 3.5 · `scripts/set_catalog.sh`

In [8]:
show("scripts/set_catalog.sh")

#!/usr/bin/env bash
# Point the space JSON at a target's FreshCart catalog before deploying to that target.
#
# The JSON is written with the dev catalog (freshcart_dev.gold..., freshcart_dev.semantic...), exactly as
# `bundle generate` exports it from dev, and bundles do not substitute variables inside that file. This
# rewrites the catalog to the target's `catalog` variable from databricks.yml. CI runs it on a fresh
# checkout; if you run it locally, don't commit the result (`git checkout src/` undoes it).
#
#     scripts/set_catalog.sh qa
set -euo pipefail

target="${1:?usage: scripts/set_catalog.sh <target>}"
space_file="src/freshcart_assistant.geniespace.json"

catalog="$(databricks bundle validate -t "$target" -o json | jq -r '.variables.catalog.value')"
if [[ -z "$catalog" || "$catalog" == "null" ]]; then
  echo "no catalog variable for target $target" >&2
  exit 1
fi
# (written to a temporary file rather than with sed -i, so it works with GNU and macOS sed alike)
sed -E "s/fresh

**Why it exists.** A bundle resolves `${var.catalog}` in YAML, but **not inside the `file_path` JSON**: the file is
sent to Databricks as it is (Part 4 shows this with the real CLI). The JSON says `freshcart_dev`, so without this
script QA and prod would read the dev data. The script asks the bundle for the target's `catalog` value
(`bundle validate -o json`) and rewrites `freshcart_dev.` in the pipeline's throw-away checkout. The file in git is never
changed, so exporting from dev stays a clean diff.

### 3.6 · `tests/test_genie.py`: the smoke test

In [9]:
show("tests/test_genie.py")

"""
Promotion-gate smoke test for the FreshCart Genie space.

Run this against a deployed space (after `bundle deploy` and the FreshCart data bundle) before
promoting it. It asks a few of the space's questions through the Genie Conversation API and asserts
each one returns a SQL-backed answer with at least one row — a cheap guard against a space that
deploys but can't actually answer (bad table refs, empty data, missing grants, broken instructions).

Usage:
    # Resolve the deployed space id from the bundle, then run the test:
    export GENIE_SPACE_ID="$(databricks bundle summary -t dev -o json \
        | jq -r '.resources.genie_spaces.freshcart_assistant.id')"
    python tests/test_genie.py          # or: pytest tests/test_genie.py

Auth comes from your active Databricks CLI profile / DATABRICKS_* env vars,
exactly like the bundle commands. Requires: pip install databricks-sdk
"""

import os
import sys

from databricks.sdk import WorkspaceClient

# Questions that have a trusted exa

The **smoke test** answers one question: *can the deployed space answer at all?* It asks three questions through the
Genie **Conversation API**, as the pipeline's service principal, and requires each answer to contain SQL **and return
at least one row** (it fetches the query result to count). It catches a space that deploys but can't work: a table
that doesn't exist in that environment, missing grants, empty data, a broken instruction.

It does **not** check that the numbers are right. That is the benchmark gate's job.

### 3.7 · `tests/benchmark_gate.py`: the benchmark gate

In [10]:
show("tests/benchmark_gate.py")

"""
Benchmark gate for the FreshCart Genie space: run every benchmark question of a deployed space
(Genie benchmark evaluation) and fail unless enough of them are answered correctly.

Benchmarks Genie cannot grade automatically come back as NEEDS_REVIEW; they are reported but do not
count towards the accuracy. The gate fails when:
  - the evaluation does not finish (or finishes with an error), or
  - no benchmark could be graded, or
  - accuracy on the graded benchmarks is below ACCURACY_THRESHOLD (default 0.60).

Usage:
    export GENIE_SPACE_ID="$(databricks bundle summary -t qa -o json \
        | jq -r '.resources.genie_spaces.freshcart_assistant.id')"
    python tests/benchmark_gate.py

Auth comes from your active Databricks CLI profile / DATABRICKS_* env vars,
exactly like the bundle commands. Requires: pip install databricks-sdk
"""

import os
import sys
import time

from databricks.sdk import WorkspaceClient

TIMEOUT_S = 1800
POLL_S = 10


def _value(x):
    """SDK enums -> the

**Genie benchmark evaluation** is a Databricks feature: it asks Genie every benchmark question, runs Genie's SQL and
the expected SQL, and compares the results. Each question gets an **assessment**:

| Assessment | Meaning | Counts in accuracy? |
|---|---|---|
| `GOOD` | Genie's result matches the expected result | yes, as correct |
| `BAD` | it doesn't (a reason such as `SINGLE_CELL_DIFFERENCE` or `RESULT_MISSING_ROWS` says how) | yes, as wrong |
| `NEEDS_REVIEW` | it couldn't be compared automatically; a person should look | no |

**accuracy = GOOD / (GOOD + BAD)**. The gate fails if the run doesn't finish, if nothing could be graded, or if accuracy
is below `ACCURACY_THRESHOLD` (60% in the workflow). Choosing the threshold is a business decision: with 5 graded
benchmarks, 60% tolerates 2 wrong answers, 90% tolerates none.

### 3.8 · `.github/workflows/deploy.yml`: the workflow

In [11]:
show(".github/workflows/deploy.yml")

# =============================================================================
# Genie-as-Code CI/CD — FreshCart Sales & Stock Assistant (simple version)
#
# Trigger map:
#   PR opened/updated            → validate only (catches schema errors before merge)
#   Push to main                 → validate → deploy dev + smoke test → deploy QA + benchmark gate
#   Manual dispatch, target=dev  → validate → dev
#   Manual dispatch, target=qa   → validate → dev → QA
#   Manual dispatch, target=prod → validate → dev → QA → prod (prod waits for approval)
#   Prod is only ever deployed by a manual dispatch, and only after the QA gate passed
#   on the same commit.
#
# Auth: one service principal per environment, stored as GitHub secrets
# (DEV_/QA_/PROD_SP_CLIENT_ID, DEV_/QA_/PROD_SP_CLIENT_SECRET) and the workspace URLs
# as GitHub variables (DEV_/QA_/PROD_DATABRICKS_HOST). The CLI and the SDK pick up
# DATABRICKS_HOST + CLIENT_ID + CLIENT_SECRET automatically.
# Pull requests from forks get no s

**How GitHub Actions reads this file.** `on:` lists the **triggers**. Each entry under `jobs:` runs on a fresh virtual
machine with a fresh copy of the repository (`actions/checkout`). `needs:` orders the jobs and `if:` decides whether a
job runs for this trigger; a job whose `needs` didn't succeed is skipped. `environment:` links a job to a **GitHub
environment** (dev, qa, prod), which can hold its own secrets and **require an approval** before the job starts. `env:`
gives the job its workspace URL and service principal credentials; the CLI and the SDK read them automatically.

Which jobs run for which trigger, computed from the file itself:

In [12]:
WF = yaml.safe_load((SIMPLE / ".github/workflows/deploy.yml").read_text())
WF["on"] = WF.pop(True, WF.get("on"))            # YAML reads the key `on` as the boolean True

def job_runs(job, event, target):
    """Evaluate a job's `if:` for an event (the simple expressions this workflow uses)."""
    expr = job.get("if")
    if not expr:
        return True
    py = expr.replace("&&", " and ").replace("||", " or ")
    return bool(eval(py, {"__builtins__": {}}, {"github": SimpleNamespace(event_name=event),
                                                  "inputs": SimpleNamespace(target=target)}))

def jobs_for(wf, event, target=None):
    running = []
    for name, job in wf["jobs"].items():                       # written in dependency order
        needs = job.get("needs", [])
        needs = [needs] if isinstance(needs, str) else needs
        if all(n in running for n in needs) and job_runs(job, event, target):
            running.append(name)
    return running

triggers = [("pull request", "pull_request", None), ("push to main", "push", None),
            ("manual run, target dev", "workflow_dispatch", "dev"), ("manual run, target qa", "workflow_dispatch", "qa"),
            ("manual run, target prod", "workflow_dispatch", "prod")]
pd.DataFrame([{"trigger": label, **{job: ("✅" if job in jobs_for(WF, ev, tg) else "—") for job in WF["jobs"]}}
              for label, ev, tg in triggers]).set_index("trigger")

,validate,deploy-dev,deploy-qa,deploy-prod
trigger,,,,
pull request,✅,—,—,—
push to main,✅,✅,✅,—
"manual run, target dev",✅,✅,—,—
"manual run, target qa",✅,✅,✅,—
"manual run, target prod",✅,✅,✅,✅


And the steps of each job:

In [13]:
rows = []
for name, job in WF["jobs"].items():
    for step in job["steps"]:
        what = step.get("uses") or " ".join(step["run"].split())
        rows.append({"job": name, "environment": job.get("environment", ""), "step": step.get("name", ""),
                     "runs": what if len(what) < 110 else what[:107] + "..."})
pd.DataFrame(rows)

,job,environment,step,runs
0,validate,,,actions/checkout@v4
1,validate,,,databricks/setup-cli@v1.18.0
2,validate,,Space JSON is valid,python3 -m json.tool src/freshcart_assistant.geniespace.json > /dev/null
3,validate,,bundle validate (dev),databricks bundle validate -t dev
4,deploy-dev,dev,,actions/checkout@v4
5,deploy-dev,dev,,databricks/setup-cli@v1.18.0
6,deploy-dev,dev,,actions/setup-python@v5
7,deploy-dev,dev,,pip install databricks-sdk --quiet
8,deploy-dev,dev,bundle deploy -t dev,databricks bundle deploy -t dev
9,deploy-dev,dev,Smoke test (conversation API),export GENIE_SPACE_ID=$(databricks bundle summary -t dev -o json \ | jq -r '.resources.genie_spaces.freshca...


| Job | Why it is there |
|---|---|
| `validate` | the cheapest check, on every pull request: the JSON parses and the bundle is valid (unknown fields, unresolved variables, a warehouse that doesn't exist...). Make it a **required check** so nothing broken can be merged |
| `deploy-dev` | deploys the merged version to the CI's dev copy and runs the **smoke test**: the first time the space meets a real workspace |
| `deploy-qa` | points the JSON at `freshcart_qa`, deploys to QA, runs the **benchmark gate** on QA data |
| `deploy-prod` | only on a manual run with target `prod`, only after the QA gate passed **in the same run**, and only after a reviewer approved the `prod` environment. Then deploys and smoke-tests prod |

`concurrency` makes runs queue instead of overlapping, so two deployments never write to the same space at once.

---
# 4 · One definition, three environments

`databricks bundle validate -t <target> -o json` shows what the bundle would deploy to each target, with every
variable resolved. We run it against a stand-in workspace (Part 5 explains the stand-ins).

In [14]:
from fake_workspace import FakeWorkspace, FreshCartEvaluator

def clean(text):
    """Drop CLI hints that don't matter here (OIDC discovery on a local server, optional AI tools)."""
    return "\n".join(l for l in text.splitlines()
                     if "Host metadata" not in l and "skills are not installed" not in l).strip()

probe = FakeWorkspace.start()
probe_env = {**{k: v for k, v in os.environ.items() if not k.startswith("DATABRICKS_")},
             "DATABRICKS_HOST": probe.host, "DATABRICKS_TOKEN": "local"}

def resolved(bundle_dir, target):
    out = subprocess.run([CLI, "bundle", "validate", "-t", target, "-o", "json"], cwd=bundle_dir,
                         env=probe_env, capture_output=True, text=True, check=True).stdout
    return json.loads(out)

rows = []
for target in ("dev", "qa", "prod"):
    cfg = resolved(SIMPLE, target)
    gs = cfg["resources"]["genie_spaces"]["freshcart_assistant"]
    tables = json.loads(gs["serialized_space"])["data_sources"]["tables"]
    rows.append({"target": target, "mode": cfg["bundle"].get("mode"), "space title": gs["title"],
                 "catalog variable": cfg["variables"]["catalog"]["value"],
                 "catalog the space reads": tables[0]["identifier"].split(".")[0],
                 "permissions": ", ".join(f'{p["level"]} {p.get("group_name") or p.get("user_name")}' for p in gs["permissions"]),
                 "root path": cfg["workspace"]["root_path"]})
pd.DataFrame(rows).set_index("target")

,mode,space title,catalog variable,catalog the space reads,permissions,root path
target,,,,,,
dev,development,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,freshcart_dev,"CAN_RUN freshcart-business-users, CAN_MANAGE dana@freshcart.example",/Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev
qa,production,FreshCart Sales & Stock Assistant [QA],freshcart_qa,freshcart_dev,"CAN_RUN freshcart-business-users, CAN_MANAGE dana@freshcart.example",/Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/qa
prod,production,FreshCart Sales & Stock Assistant,freshcart,freshcart_dev,"CAN_RUN freshcart-business-users, CAN_MANAGE dana@freshcart.example",/Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/prod


Look at the last two catalog columns: the variable is right for every target, but **the space reads `freshcart_dev`
in all three**, because variables are not resolved inside the JSON file. That is the problem `set_catalog.sh` solves.
Here it is, run the way the pipeline runs it, on a throw-away copy:

In [15]:
rows = []
for target in ("qa", "prod"):
    copy_dir = Path(tempfile.mkdtemp()) / "checkout"
    shutil.copytree(SIMPLE, copy_dir, ignore=shutil.ignore_patterns(".databricks", "__pycache__"))
    env = {**probe_env, "PATH": f"{Path(CLI).parent}{os.pathsep}{os.environ['PATH']}"}
    if Path(CLI).name != "databricks":                     # the script calls `databricks`
        (copy_dir.parent / "bin").mkdir(); (copy_dir.parent / "bin" / "databricks").symlink_to(CLI)
        env["PATH"] = f"{copy_dir.parent / 'bin'}{os.pathsep}{env['PATH']}"
    out = subprocess.run(["bash", "scripts/set_catalog.sh", target], cwd=copy_dir, env=env,
                         capture_output=True, text=True, check=True).stdout
    gs = resolved(copy_dir, target)["resources"]["genie_spaces"]["freshcart_assistant"]
    idents = sorted({t["identifier"] for t in json.loads(gs["serialized_space"])["data_sources"]["tables"]})
    rows.append({"target": target, "script says": out.strip(), "tables the space reads now": ", ".join(idents)})
    shutil.rmtree(copy_dir.parent)
probe.stop()
pd.DataFrame(rows).set_index("target")

,script says,tables the space reads now
target,,
qa,Space for target qa now reads catalog freshcart_qa,"freshcart_qa.gold.dim_product, freshcart_qa.gold.dim_store"
prod,Space for target prod now reads catalog freshcart,"freshcart.gold.dim_product, freshcart.gold.dim_store"


> **Why not keep `${var.catalog}` in the file?** The CLI would send the literal text `${var.catalog}` to Databricks.
> The full bundle (`genie_bundle/`) solves it differently: it stores the space as a bundle *variable* in YAML, where
> `${var.catalog}` **is** resolved, and pays for it with a conversion script on every export. The simple bundle keeps
> the exported JSON untouched and rewrites one prefix at deploy time.

---
# 5 · Hands-on: the full lifecycle

### 5.1 · The lab

We start **three stand-in workspaces**, one each for dev, qa and prod (`tests/fake_workspace.py`). They are small local
servers that implement the Databricks REST endpoints the bundle and the tests use: workspace files, Genie spaces,
permissions, Genie conversations and benchmark runs.

| Real | In this lab |
|---|---|
| Databricks CLI, `bundle validate / deploy / plan / summary / generate` | **the real CLI** |
| the bundle files and `deploy.yml` | **the real files**, in a scratch git repository |
| the smoke test and the benchmark gate | **the real scripts**, using the real Databricks SDK |
| GitHub Actions | `github_actions()` below: reads `deploy.yml` and runs its jobs in order, each on a fresh checkout of the commit, each `run:` step with `bash -eo pipefail` and the job's `env`, like the GitHub runner |
| Genie (the AI) | **not real**: the stand-in answers a question with the space's trusted example SQL for that exact question, runs it on the local FreshCart data, and grades a benchmark by comparing results (no example → `NEEDS_REVIEW`) |
| a service principal per environment, OAuth | one stand-in user and a token (the stand-ins have no OAuth) |
| reviewers approving the `prod` environment | printed as an approval |

Because every identity is the same stand-in user here, the CI's dev copy and Dana's own dev copy are the same space.
On Databricks the service principal gets its own `[dev <service principal>]` copy.

In [16]:
LAB = Path(tempfile.mkdtemp(prefix="genie_simple_lab_"))
REPO = LAB / "repo"                                    # the bundle folder as the root of its own repository
shutil.copytree(SIMPLE, REPO, ignore=shutil.ignore_patterns(".databricks", "__pycache__"))
(REPO / ".gitignore").write_text((SIMPLE / ".gitignore").read_text())

evaluator = FreshCartEvaluator()
WS = {env: FakeWorkspace.start(evaluator=evaluator) for env in ("dev", "qa", "prod")}

BIN = LAB / "bin"; BIN.mkdir()
(BIN / "databricks").symlink_to(CLI)                   # what databricks/setup-cli provides on the runner
BASE_ENV = {k: v for k, v in os.environ.items() if not k.startswith("DATABRICKS_")}
BASE_ENV["PATH"] = os.pathsep.join([str(BIN), str(Path(sys.executable).parent), os.environ["PATH"]])  # + setup-python

def tidy(text):
    text = clean(text).replace(str(LAB), "$LAB")
    for env, ws in WS.items():
        text = text.replace(ws.host, f"https://{env}-workspace")
    return text

def git(*args, show=True):
    out = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True, check=True).stdout.strip()
    if show:
        if out:
            print(out)
        return None
    return out

git("init", "-q", "-b", "main")
git("config", "user.name", "Dana Developer"); git("config", "user.email", "dana@freshcart.example")
git("add", "-A"); git("commit", "-q", "-m", "FreshCart Genie space as code")
git("log", "--oneline")
for env, ws in WS.items():
    print(f"{env:<5} stand-in workspace running (shown as https://{env}-workspace)")

8cf3f20 FreshCart Genie space as code
dev   stand-in workspace running (shown as https://dev-workspace)
qa    stand-in workspace running (shown as https://qa-workspace)
prod  stand-in workspace running (shown as https://prod-workspace)


The **GitHub Actions stand-in**. Read it once: it is a faithful, small model of what the runner does with `deploy.yml`.

In [17]:
APPROVER = "Ravi (release manager)"

def checkout(ref):
    """actions/checkout: a fresh copy of the repository at that commit, in a new folder."""
    d = Path(tempfile.mkdtemp(dir=LAB, prefix="runner_"))
    archive = subprocess.run(["git", "-C", str(REPO), "archive", ref], capture_output=True, check=True).stdout
    subprocess.run(["tar", "-x", "-C", str(d)], input=archive, check=True)
    return d

def step_env(job, step):
    """The job's and step's env, with ${{ vars.X_DATABRICKS_HOST }} and ${{ secrets.* }} filled in like GitHub does."""
    env = dict(BASE_ENV)
    for k, v in {**job.get("env", {}), **step.get("env", {})}.items():
        v = re.sub(r"\$\{\{\s*vars\.(\w+)_DATABRICKS_HOST\s*\}\}", lambda m: WS[m.group(1).lower()].host, str(v))
        env[k] = re.sub(r"\$\{\{\s*secrets\.\w+\s*\}\}", "stand-in-secret", v)
    if env.pop("DATABRICKS_CLIENT_ID", None):           # the stand-ins have no OAuth: use a token instead
        env.pop("DATABRICKS_CLIENT_SECRET", None)
        env["DATABRICKS_TOKEN"] = "stand-in-token"
    return env

def run_step(workdir, job, step):
    p = subprocess.run(["bash", "--noprofile", "--norc", "-eo", "pipefail", "-c", step["run"]], cwd=workdir,
                       env=step_env(job, step), capture_output=True, text=True)
    return p.returncode == 0, tidy(p.stdout + "\n" + p.stderr)

def github_actions(event, target=None, ref="main", brief=False):
    """Run .github/workflows/deploy.yml for one trigger, the way GitHub Actions does. Returns {job: result}.
    brief=True prints only the last line of each successful step (a failing step always prints everything)."""
    sha = git("rev-parse", "--short", ref, show=False)
    subject = git("log", "-1", "--format=%s", ref, show=False)
    print(f"▶ workflow 'Deploy Genie Space' · {event}{f' (target: {target})' if target else ''} · {ref} @ {sha} “{subject}”")
    wf = yaml.safe_load(subprocess.run(["git", "-C", str(REPO), "show", f"{ref}:.github/workflows/deploy.yml"],
                                       capture_output=True, text=True, check=True).stdout)
    results = {}
    for name, job in wf["jobs"].items():
        needs = job.get("needs", [])
        needs = [needs] if isinstance(needs, str) else needs
        if not all(results.get(n) == "success" for n in needs) or not job_runs(job, event, target):
            results[name] = "skipped"
            print(f"\n  ⏭  {name}: skipped")
            continue
        print(f"\n  ▶ {name} · {job['name']}" + (f" · environment {job['environment']}" if job.get("environment") else ""))
        if job.get("environment") == "prod":
            print(f"     ⏸  environment 'prod' requires a review → approved by {APPROVER}")
        workdir, results[name] = checkout(ref), "success"
        for step in job["steps"]:
            if "uses" in step:
                print(f"     ·  {step['uses']}")
                continue
            ok, out = run_step(workdir, job, step)
            print(f"     {'✔' if ok else '✘'}  {step.get('name') or step['run'].strip()}")
            if out and brief and ok:
                out = out.splitlines()[-1]
            if out:
                print(textwrap.indent(out, "        "))
            if not ok:
                results[name] = "failure"
                break
        print(f"  {'✅' if results[name] == 'success' else '❌'} {name}: {results[name]}")
    return results

Two more helpers: what is deployed where (read from the stand-in workspaces), and commands Dana types on her laptop
(in her clone of the repository, logged in to the dev workspace).

In [18]:
def spaces():
    """Every Genie space in every workspace, with what it contains."""
    rows = []
    for env, ws in WS.items():
        for sp in ws.spaces.values():
            s = json.loads(sp["serialized_space"])
            ds = s["data_sources"]
            idents = [t["identifier"] for t in ds.get("tables", []) + ds.get("metric_views", [])]
            rows.append({"workspace": env, "space title": sp["title"],
                         "catalog": ", ".join(sorted({i.split(".")[0] for i in idents})),
                         "examples": len(s["instructions"].get("example_question_sqls", [])),
                         "benchmarks": len(s.get("benchmarks", {}).get("questions", [])),
                         "sample questions": len(s["config"].get("sample_questions", [])),
                         "edits (etag)": sp["etag"]})
    return pd.DataFrame(rows)

def live_space(env):
    """The serialized space currently live in a workspace (one space per workspace in this lab)."""
    return json.loads(next(iter(WS[env].spaces.values()))["serialized_space"])

def ui_edit(env, change):
    """Simulate someone editing the space in the Genie UI of a workspace."""
    WS[env].ui_edit(next(iter(WS[env].spaces)), change)

def dana(*args):
    """A command Dana runs on her laptop, in her clone of the repository, against the dev workspace."""
    p = subprocess.run(list(args), cwd=REPO, env={**BASE_ENV, "DATABRICKS_HOST": WS["dev"].host,
                                                  "DATABRICKS_TOKEN": "dana-token"}, capture_output=True, text=True)
    print("$ " + " ".join(args))
    out = tidy(p.stdout + "\n" + p.stderr)
    if out:
        print(out)
    if p.returncode:
        raise RuntimeError(f"exit code {p.returncode}")

### 5.2 · The first release: push to `main`

The repository has its first commit on `main`. A push to `main` starts the workflow: **validate → dev + smoke test →
QA + benchmark gate**. Read the output from top to bottom: every line is the real CLI or one of the two test scripts.

In [19]:
result = github_actions("push")

▶ workflow 'Deploy Genie Space' · push · main @ 8cf3f20 “FreshCart Genie space as code”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid


     ✔  bundle validate (dev)
        Name: freshcart-genie-simple
        Target: dev
        Workspace:
          User: dana@freshcart.example
          Path: /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev

        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev/files...
        Created genie_spaces.freshcart_assistant
        Created genie_spaces.freshcart_assistant.permissions
        Files: 9 uploaded, 0 deleted
        Resources: 2 created, 0 changed, 0 deleted, 0 unchanged


     ✔  Smoke test (conversation API)
        [PASS] What were net sales and margin by region last week?  (COMPLETED, 5 row(s))
        [PASS] Which categories rely most on promotions this quarter?  (COMPLETED, 17 row(s))
        [PASS] Where are we out of stock most often in fresh?  (COMPLETED, 10 row(s))

        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


     ✔  bundle deploy -t qa
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/qa/files...
        Created genie_spaces.freshcart_assistant
        Created genie_spaces.freshcart_assistant.permissions
        Files: 9 uploaded, 0 deleted
        Resources: 2 created, 0 changed, 0 deleted, 0 unchanged


     ✔  Genie benchmark gate
        Eval run started: 39825bd7c833446da82be1c651c35396

        Benchmark results: 4/4 graded correct (100%), 4 need review
          ❔ How much Produce stock did we have at the end of last week?
          ❔ What were total sales last week?
          ✅ How are we trading year to date versus last year?
          ❔ What is our like-for-like growth by region this year?
          ❔ What is gross margin % by region YTD?
          ✅ What were net sales and margin by region last week?
          ✅ Which categories rely most on promotions this quarter?
          ✅ Where are we out of stock most often in fresh?

        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ⏭  deploy-prod: skipped


What happened:

- **validate** parsed the JSON and validated the bundle against the dev workspace (`Validation OK!`).
- **deploy-dev** created the space (`Created genie_spaces.freshcart_assistant`) and its permissions, then the smoke
  test asked three questions; each came back with SQL and rows.
- **deploy-qa** rewrote the catalog to `freshcart_qa`, deployed, and ran the benchmarks: the four questions that have a
  trusted example were graded `GOOD` (✅), the four without one need a human review (❔), so accuracy is 4/4.
- **deploy-prod** was skipped: a push never deploys prod.

In [20]:
spaces()

,workspace,space title,catalog,examples,benchmarks,sample questions,edits (etag)
0,dev,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,4,8,4,1
1,qa,FreshCart Sales & Stock Assistant [QA],freshcart_qa,4,8,4,1


### 5.3 · Promote to prod: a manual run

In GitHub: *Actions → Deploy Genie Space → Run workflow*, branch `main`, target `prod`. The run re-checks dev and QA on
this exact commit, then waits for the `prod` environment's reviewer, then deploys prod and smoke-tests it.

In [21]:
result = github_actions("workflow_dispatch", "prod")

▶ workflow 'Deploy Genie Space' · workflow_dispatch (target: prod) · main @ 8cf3f20 “FreshCart Genie space as code”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Name: freshcart-genie-simple
        Target: dev
        Workspace:
          User: dana@freshcart.example
          Path: /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev

        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev


     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev/files...
        Files: 9 uploaded, 0 deleted
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Smoke test (conversation API)
        [PASS] What were net sales and margin by region last week?  (COMPLETED, 5 row(s))
        [PASS] Which categories rely most on promotions this quarter?  (COMPLETED, 17 row(s))
        [PASS] Where are we out of stock most often in fresh?  (COMPLETED, 10 row(s))

        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


...

     ✔  bundle deploy -t qa
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/qa/files...
        Files: 9 uploaded, 0 deleted
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Genie benchmark gate
        Eval run started: dc9c7ef88c36458eb2eb88f5327bc618

        Benchmark results: 4/4 graded correct (100%), 4 need review
          ❔ How much Produce stock did we have at the end of last week?
          ❔ What were total sales last week?
          ✅ How are we trading year to date versus last year?
          ❔ What is our like-for-like growth by region this year?
          ❔ What is gross margin % by region YTD?
          ✅ What were net sales and margin by region last week?
          ✅ Which categories rely most on promotions this quarter?
          ✅ Where are we out of stock most often in fresh?

        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ▶ deploy-prod · Deploy → prod · environment prod
     ⏸  environment 'prod' requires a review → approved by Ravi (release manager)
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the prod catalog
        Space for target prod now reads catalog freshcart


     ✔  bundle deploy -t prod
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/prod/files...
        Created genie_spaces.freshcart_assistant
        Created genie_spaces.freshcart_assistant.permissions
        Files: 9 uploaded, 0 deleted
        Resources: 2 created, 0 changed, 0 deleted, 0 unchanged


     ✔  Smoke test (conversation API)
        [PASS] What were net sales and margin by region last week?  (COMPLETED, 5 row(s))
        [PASS] Which categories rely most on promotions this quarter?  (COMPLETED, 17 row(s))
        [PASS] Where are we out of stock most often in fresh?  (COMPLETED, 10 row(s))

        All Genie smoke questions answered — safe to promote.
  ✅ deploy-prod: success


Notice `Resources: 0 created, 0 changed, ... unchanged` for dev and QA: a deployment is **idempotent**; deploying
the same commit again changes nothing. Prod was created on the `freshcart` catalog:

In [22]:
spaces()

,workspace,space title,catalog,examples,benchmarks,sample questions,edits (etag)
0,dev,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,4,8,4,1
1,qa,FreshCart Sales & Stock Assistant [QA],freshcart_qa,4,8,4,1
2,prod,FreshCart Sales & Stock Assistant,freshcart,4,8,4,1


### 5.4 · The authoring loop: from the dev UI to prod

Category managers keep asking about online sales. Dana improves the **dev** space **in the Genie UI**: she adds a
trusted example for *"What share of sales is online, by region, this year?"*, a benchmark for it, and a sample
question. (Here the stand-in applies the edit; in real life she clicks through the Genie UI.)

In [23]:
ONLINE_Q = "What share of sales is online, by region, this year?"
ONLINE_SQL = ["SELECT region, MEASURE(online_sales_share) AS online_sales_share\n",
              "FROM freshcart_dev.semantic.sales_metrics\n",
              "WHERE is_fiscal_ytd\n",
              "GROUP BY ALL\n",
              "ORDER BY region"]

def dana_adds_online_share(s):
    s["instructions"]["example_question_sqls"].append({"id": "5a1e0c3d9b7f4e2a8c6d1f0b3e5a7c9d", "question": [ONLINE_Q], "sql": ONLINE_SQL})
    s["benchmarks"]["questions"].append({"id": "eb2f4863dd92b072d956e8d2c685923d", "question": [ONLINE_Q],
                                         "answer": [{"format": "SQL", "content": ONLINE_SQL}]})
    s["config"]["sample_questions"].append({"id": "a3c5e7f9b1d34f6a8c0e2b4d6f8a0c1e", "question": [ONLINE_Q]})
    for lst in (s["instructions"]["example_question_sqls"], s["benchmarks"]["questions"], s["config"]["sample_questions"]):
        lst.sort(key=lambda item: item["id"])             # Databricks keeps these lists sorted by id

ui_edit("dev", dana_adds_online_share)
print("the dev space now has", len(live_space("dev")["instructions"]["example_question_sqls"]), "examples;",
      "git still has", len(json.loads((REPO / "src/freshcart_assistant.geniespace.json").read_text())["instructions"]["example_question_sqls"]))

the dev space now has 5 examples; git still has 4


The UI edit exists **only in the dev workspace**. Until it is in git it has no history, no review, and the next
deployment to dev would overwrite it. Dana captures it with **`bundle generate`** on a branch:

In [24]:
git("checkout", "-q", "-b", "feature/online-share")
dana("databricks", "bundle", "generate", "genie-space", "--resource", "freshcart_assistant", "--force", "-t", "dev")
git("status", "--short")

$ databricks bundle generate genie-space --resource freshcart_assistant --force -t dev
Writing genie space to src/freshcart_assistant.geniespace.json
M src/freshcart_assistant.geniespace.json


Only the JSON changed, and the diff shows exactly what she did in the UI. This is what reviewers read in the pull
request:

In [25]:
git("diff")

diff --git a/src/freshcart_assistant.geniespace.json b/src/freshcart_assistant.geniespace.json
index 59df0af..9a1a56f 100644
--- a/src/freshcart_assistant.geniespace.json
+++ b/src/freshcart_assistant.geniespace.json
@@ -151,6 +151,24 @@
         "question": [
           "Where are we out of stock most often in fresh?"
         ]
+      },
+      {
+        "answer": [
+          {
+            "content": [
+              "SELECT region, MEASURE(online_sales_share) AS online_sales_share\n",
+              "FROM freshcart_dev.semantic.sales_metrics\n",
+              "WHERE is_fiscal_ytd\n",
+              "GROUP BY ALL\n",
+              "ORDER BY region"
+            ],
+            "format": "SQL"
+          }
+        ],
+        "id": "eb2f4863dd92b072d956e8d2c685923d",
+        "question": [
+          "What share of sales is online, by region, this year?"
+        ]
       }
     ]
   },
@@ -174,6 +192,12 @@
           "How are we trading year to date versus last year?"
         

She commits, pushes the branch and opens a pull request. The workflow runs **validate** only:

In [26]:
git("commit", "-q", "-am", "Add a trusted example and a benchmark for online share by region")
result = github_actions("pull_request", ref="feature/online-share")

▶ workflow 'Deploy Genie Space' · pull_request · feature/online-share @ cb2cac8 “Add a trusted example and a benchmark for online share by region”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Name: freshcart-genie-simple
        Target: dev
        Workspace:
          User: dana@freshcart.example
          Path: /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev

        Validation OK!
  ✅ validate: success

  ⏭  deploy-dev: skipped

  ⏭  deploy-qa: skipped

  ⏭  deploy-prod: skipped


Validation passed and a colleague approves the pull request. Merging it is a push to `main`: dev and QA are updated,
and QA now grades **five** benchmarks, including Dana's new one. Then the release manager promotes it to prod.

In [27]:
git("checkout", "-q", "main")
git("merge", "-q", "--no-ff", "feature/online-share", "-m", "Merge pull request #1 from feature/online-share")
result = github_actions("push")

▶ workflow 'Deploy Genie Space' · push · main @ 796da12 “Merge pull request #1 from feature/online-share”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Name: freshcart-genie-simple
        Target: dev
        Workspace:
          User: dana@freshcart.example
          Path: /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev

        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/dev/files...
        Updated genie_spaces.freshcart_assistant
        Files: 9 uploaded, 0 deleted
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        [PASS] What were net sales and margin by region last week?  (COMPLETED, 5 row(s))
        [PASS] Which categories rely most on promotions this quarter?  (COMPLETED, 17 row(s))
        [PASS] Where are we out of stock most often in fresh?  (COMPLETED, 10 row(s))

        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


     ✔  bundle deploy -t qa
        Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie-simple/qa/files...
        Updated genie_spaces.freshcart_assistant
        Files: 9 uploaded, 0 deleted
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Genie benchmark gate
        Eval run started: 951c30d83bce4c308d1e154014c6c923

        Benchmark results: 5/5 graded correct (100%), 4 need review
          ❔ How much Produce stock did we have at the end of last week?
          ❔ What were total sales last week?
          ✅ How are we trading year to date versus last year?
          ❔ What is our like-for-like growth by region this year?
          ❔ What is gross margin % by region YTD?
          ✅ What were net sales and margin by region last week?
          ✅ Which categories rely most on promotions this quarter?
          ✅ Where are we out of stock most often in fresh?
          ✅ What share of sales is online, by region, this year?

        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ⏭  deploy-prod: skipped


In [28]:
result = github_actions("workflow_dispatch", "prod", brief=True)
spaces()

▶ workflow 'Deploy Genie Space' · workflow_dispatch (target: prod) · main @ 796da12 “Merge pull request #1 from feature/online-share”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev


     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


...

     ✔  Genie benchmark gate
        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ▶ deploy-prod · Deploy → prod · environment prod
     ⏸  environment 'prod' requires a review → approved by Ravi (release manager)
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the prod catalog
        Space for target prod now reads catalog freshcart


     ✔  bundle deploy -t prod
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-prod: success


,workspace,space title,catalog,examples,benchmarks,sample questions,edits (etag)
0,dev,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,5,9,5,3
1,qa,FreshCart Sales & Stock Assistant [QA],freshcart_qa,5,9,5,2
2,prod,FreshCart Sales & Stock Assistant,freshcart,5,9,5,2


Every environment now runs the same version: 5 examples, 9 benchmarks, 5 sample questions, each on its own catalog.

From the promotion above on, runs are printed **briefly**: the last line of each successful step, and the full output
of any step that fails.

### 5.5 · A bad change, caught by the QA gate

Someone decides the examples should show **gross** sales and does a find-and-replace in the JSON in their editor
(hand edits are allowed; they go through the same pull request). It is valid JSON, valid SQL, and every query still
returns rows.

In [29]:
git("checkout", "-q", "-b", "feature/show-gross-sales")
path = REPO / "src/freshcart_assistant.geniespace.json"
s = json.loads(path.read_text())
for e in s["instructions"]["example_question_sqls"]:
    e["sql"] = [line.replace("MEASURE(net_sales)", "MEASURE(gross_sales)") for line in e["sql"]]
path.write_text(json.dumps(s, indent=2, sort_keys=True, ensure_ascii=False) + "\n")
git("diff", "--stat")
git("commit", "-q", "-am", "Show gross sales in the example queries")
result = github_actions("pull_request", ref="feature/show-gross-sales", brief=True)

src/freshcart_assistant.geniespace.json | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)
▶ workflow 'Deploy Genie Space' · pull_request · feature/show-gross-sales @ 5506bdb “Show gross sales in the example queries”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid


     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ⏭  deploy-dev: skipped

  ⏭  deploy-qa: skipped

  ⏭  deploy-prod: skipped


The pull request is green: **validation cannot know what the numbers should be.** The change is merged:

In [30]:
git("checkout", "-q", "main")
git("merge", "-q", "--no-ff", "feature/show-gross-sales", "-m", "Merge pull request #2 from feature/show-gross-sales")
result = github_actions("push", brief=True)

▶ workflow 'Deploy Genie Space' · push · main @ dbe8030 “Merge pull request #2 from feature/show-gross-sales”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


     ✔  bundle deploy -t qa
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✘  Genie benchmark gate
        Eval run started: df99cacbe5bd4a76ab979e7a9124ab49

        Benchmark results: 2/5 graded correct (40%), 4 need review
          ❔ How much Produce stock did we have at the end of last week?
          ❔ What were total sales last week?
          ❌ How are we trading year to date versus last year?  [SINGLE_CELL_DIFFERENCE]
          ❔ What is our like-for-like growth by region this year?
          ❔ What is gross margin % by region YTD?
          ❌ What were net sales and margin by region last week?  [SINGLE_CELL_DIFFERENCE]
          ❌ Which categories rely most on promotions this quarter?  [SINGLE_CELL_DIFFERENCE]
          ✅ Where are we out of stock most often in fresh?
          ✅ What share of sales is online, by region, this year?

        GATE FAILED: accuracy 40% < threshold 60%
  ❌ deploy-qa: failure

  ⏭  deploy-prod: skipped


Read the two test results:

- the **smoke test passed** in dev: Genie still answers with SQL and rows. It checks *can it answer*, not *is it right*;
- the **benchmark gate failed** in QA: three trusted examples now return gross sales, so their benchmarks (which expect
  net sales) are `BAD` with `SINGLE_CELL_DIFFERENCE`; accuracy fell to 40%, below 60%.

Could it still reach prod? Try the manual promotion:

In [31]:
result = github_actions("workflow_dispatch", "prod", brief=True)

▶ workflow 'Deploy Genie Space' · workflow_dispatch (target: prod) · main @ dbe8030 “Merge pull request #2 from feature/show-gross-sales”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


     ✔  bundle deploy -t qa
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✘  Genie benchmark gate
        Eval run started: 0569dedfadd84407999277881c959a88

        Benchmark results: 2/5 graded correct (40%), 4 need review
          ❔ How much Produce stock did we have at the end of last week?
          ❔ What were total sales last week?
          ❌ How are we trading year to date versus last year?  [SINGLE_CELL_DIFFERENCE]
          ❔ What is our like-for-like growth by region this year?
          ❔ What is gross margin % by region YTD?
          ❌ What were net sales and margin by region last week?  [SINGLE_CELL_DIFFERENCE]
          ❌ Which categories rely most on promotions this quarter?  [SINGLE_CELL_DIFFERENCE]
          ✅ Where are we out of stock most often in fresh?
          ✅ What share of sales is online, by region, this year?

        GATE FAILED: accuracy 40% < threshold 60%
  ❌ deploy-qa: failure

  ⏭  deploy-prod: skipped


The QA gate fails again **in the same run**, so `deploy-prod` is skipped: prod is protected. Users keep the good
version:

In [32]:
spaces()

,workspace,space title,catalog,examples,benchmarks,sample questions,edits (etag)
0,dev,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,5,9,5,4
1,qa,FreshCart Sales & Stock Assistant [QA],freshcart_qa,5,9,5,3
2,prod,FreshCart Sales & Stock Assistant,freshcart,5,9,5,2


Dev and QA run the bad version, prod doesn't. The fix is a normal change: **revert** the merge (a new commit that
undoes it, so the history keeps both) and push:

In [33]:
git("revert", "-m", "1", "--no-edit", "HEAD")
git("log", "--oneline", "-4")
result = github_actions("push", brief=True)

[main 2cdc3d3] Revert "Merge pull request #2 from feature/show-gross-sales"
 Date: Fri Oct 9 17:53:55 2026 +0000
 1 file changed, 3 insertions(+), 3 deletions(-)
2cdc3d3 Revert "Merge pull request #2 from feature/show-gross-sales"
dbe8030 Merge pull request #2 from feature/show-gross-sales
5506bdb Show gross sales in the example queries
796da12 Merge pull request #1 from feature/online-share
▶ workflow 'Deploy Genie Space' · push · main @ 2cdc3d3 “Revert "Merge pull request #2 from feature/show-gross-sales"”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


     ✔  bundle deploy -t qa
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Genie benchmark gate
        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ⏭  deploy-prod: skipped


Green again. **Lesson on thresholds:** had only *one* example been changed, accuracy would be 4/5 = 80% and the
60% gate would have **passed** with one wrong answer. The threshold is a deliberate trade-off between blocking on
Genie's normal variation and catching real mistakes; the gate prints every `BAD` question either way, so read its
output even when it passes.

### 5.6 · Rollback: a change the gates can't catch

Benchmarks check **SQL results**. They cannot judge how Genie *talks*. A change to the summary instructions passes
every gate and reaches prod:

In [34]:
OLD_RULE = "- Round percentages to one decimal place in the summary text.\n"
NEW_RULE = "- Never show numbers in the summary; describe the trend in one word.\n"

git("checkout", "-q", "-b", "feature/shorter-summaries")
path.write_text(path.read_text().replace(json.dumps(OLD_RULE), json.dumps(NEW_RULE)))
git("commit", "-q", "-am", "Shorter summaries")
git("checkout", "-q", "main")
git("merge", "-q", "--no-ff", "feature/shorter-summaries", "-m", "Merge pull request #3 from feature/shorter-summaries")
result = github_actions("push", brief=True)
result = github_actions("workflow_dispatch", "prod", brief=True)
print("\nprod summary rule:", [l for l in live_space("prod")["instructions"]["text_instructions"][0]["content"] if "summary" in l])

▶ workflow 'Deploy Genie Space' · push · main @ aa43145 “Merge pull request #3 from feature/shorter-summaries”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the QA catalog
        Space for target qa now reads catalog freshcart_qa


...

     ✔  bundle deploy -t qa
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Genie benchmark gate
        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ▶ deploy-prod · Deploy → prod · environment prod
     ⏸  environment 'prod' requires a review → approved by Ravi (release manager)
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the prod catalog
        Space for target prod now reads catalog freshcart


     ✔  bundle deploy -t prod
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-prod: success

prod summary rule: ['- Never show numbers in the summary; describe the trend in one word.\n']


Finance complains the next morning: the answers have no numbers. **Rolling back = deploying the previous good
version.** With git that is a revert of the merge, then the same pipeline as any change (the revert is itself
reviewed, validated and gated):

In [35]:
bad_merge = git("log", "-1", "--format=%h", "--grep", "Merge pull request #3", show=False)
git("revert", "-m", "1", "--no-edit", bad_merge)
result = github_actions("push", brief=True)
result = github_actions("workflow_dispatch", "prod", brief=True)
print("\nprod summary rule:", [l for l in live_space("prod")["instructions"]["text_instructions"][0]["content"] if "summary" in l])
print("prod is back to the reviewed rule:", OLD_RULE in live_space("prod")["instructions"]["text_instructions"][0]["content"])

[main 615c95e] Revert "Merge pull request #3 from feature/shorter-summaries"
 Date: Fri Oct 9 17:55:03 2026 +0000
 1 file changed, 1 insertion(+), 1 deletion(-)
▶ workflow 'Deploy Genie Space' · push · main @ 615c95e “Revert "Merge pull request #3 from feature/shorter-summaries"”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


...

     ✔  bundle deploy -t qa
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Genie benchmark gate
        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ▶ deploy-prod · Deploy → prod · environment prod
     ⏸  environment 'prod' requires a review → approved by Ravi (release manager)
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the prod catalog
        Space for target prod now reads catalog freshcart


     ✔  bundle deploy -t prod
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-prod: success

prod summary rule: ['- Round percentages to one decimal place in the summary text.\n']
prod is back to the reviewed rule: True


| Rollback option | How | When |
|---|---|---|
| **Revert the merge** (shown) | `git revert -m 1 <merge>` → push → manual run, target prod | the normal way: `main` keeps matching prod, and the history shows the rollback |
| **Deploy an older commit directly** | run the workflow on a tag or branch that points at the old commit ("Use workflow from") | an emergency, when even the revert's pull request is too slow; then revert on `main` too, or the next release brings the bad change back |

A rollback takes as long as one pipeline run; it needs no access to prod, only to GitHub.

### 5.7 · Someone edits prod in the UI: drift

Rule 3 says nobody edits prod by hand. Suppose an analyst with too many permissions adds an instruction directly in
the prod space:

In [36]:
WEATHER = "Always mention the weather."
ui_edit("prod", lambda s: s["instructions"]["text_instructions"][0]["content"].append("\n" + WEATHER))
print("edit live in prod:", any(WEATHER in l for l in live_space("prod")["instructions"]["text_instructions"][0]["content"]))

edit live in prod: True


Prod no longer matches git: this is **drift**. `bundle plan` (what `deploy` *would* do) notices that the live space
differs from the bundle, run here exactly as the prod job would (fresh checkout, prod catalog):

In [37]:
prod_job = WF["jobs"]["deploy-prod"]
workdir = checkout("main")
for script in ("bash scripts/set_catalog.sh prod", "databricks bundle plan -t prod"):
    ok, out = run_step(workdir, prod_job, {"run": script})
    print(f"$ {script}\n{textwrap.indent(out, '  ')}\n")

$ bash scripts/set_catalog.sh prod
  Space for target prod now reads catalog freshcart



$ databricks bundle plan -t prod
  update genie_spaces.freshcart_assistant

  Plan: 0 to add, 1 to change, 0 to delete, 1 unchanged



`1 to change` means the next deployment **overwrites** the space with git's version. The plan can't tell you whether
the change is yours (a new release) or someone's UI edit. The next release does exactly that, silently:

In [38]:
result = github_actions("workflow_dispatch", "prod", brief=True)
print("\nedit still live in prod:", any(WEATHER in l for l in live_space("prod")["instructions"]["text_instructions"][0]["content"]))

▶ workflow 'Deploy Genie Space' · workflow_dispatch (target: prod) · main @ 615c95e “Revert "Merge pull request #3 from feature/shorter-summaries"”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✔  Space JSON is valid
     ✔  bundle validate (dev)
        Validation OK!
  ✅ validate: success

  ▶ deploy-dev · Deploy → dev + smoke test · environment dev
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet


     ✔  bundle deploy -t dev
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-dev: success

  ▶ deploy-qa · Deploy → QA + benchmark gate · environment qa
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


...

     ✔  bundle deploy -t qa
        Resources: 0 created, 0 changed, 0 deleted, 2 unchanged


     ✔  Genie benchmark gate
        Gate passed — deploying to prod is safe.
  ✅ deploy-qa: success

  ▶ deploy-prod · Deploy → prod · environment prod
     ⏸  environment 'prod' requires a review → approved by Ravi (release manager)
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0
     ·  actions/setup-python@v5


     ✔  pip install databricks-sdk --quiet
     ✔  Point the space at the prod catalog
        Space for target prod now reads catalog freshcart


     ✔  bundle deploy -t prod
        Resources: 0 created, 1 changed, 0 deleted, 1 unchanged


     ✔  Smoke test (conversation API)
        All Genie smoke questions answered — safe to promote.
  ✅ deploy-prod: success

edit still live in prod: False


The prod edit is gone, and that is correct (git is the source of truth), but nobody was told. How to handle drift
with the simple bundle:

| | |
|---|---|
| **Prevent it** (the real fix) | in qa and prod only the deploying service principal has `CAN_MANAGE` on the space; business users get `CAN_RUN`. Keep workspace admins out of the habit of editing spaces |
| **Adopt a good prod edit** | make the same change in the **dev** UI, `bundle generate`, pull request. Never export from prod: the file would then say `freshcart` and dev would read prod data |
| **Detect it automatically** | that is what the full bundle adds (`genie_bundle/scripts/check_drift.py`): it reads `bundle plan -o json`, stops the release when the live space was edited since the last deployment, and saves a backup of it |

### 5.8 · A broken pull request

A hand edit that leaves the JSON broken (here: a stray comma) never gets near a workspace. The `validate` job fails
on the pull request, and with `validate` as a required check, GitHub won't let anyone merge it:

In [39]:
git("checkout", "-q", "-b", "feature/typo")
path.write_text(path.read_text().replace('"version": 2', '"version": 2,', 1))
git("commit", "-q", "-am", "Tweak the space by hand")
result = github_actions("pull_request", ref="feature/typo", brief=True)
git("checkout", "-q", "main"); git("branch", "-q", "-D", "feature/typo")

▶ workflow 'Deploy Genie Space' · pull_request · feature/typo @ 2a5f76c “Tweak the space by hand”

  ▶ validate · Validate bundle
     ·  actions/checkout@v4
     ·  databricks/setup-cli@v1.18.0


     ✘  Space JSON is valid
        Illegal trailing comma before end of object: line 416 column 15 (char 14498)
  ❌ validate: failure

  ⏭  deploy-dev: skipped

  ⏭  deploy-qa: skipped

  ⏭  deploy-prod: skipped


### 5.9 · The audit trail

Everything that happened is in git: every change, the merge that brought it in, the reverts. Together with the
workflow runs in GitHub (who triggered, who approved prod, which tests ran), that answers *what is running in prod,
since when, and why*.

In [40]:
git("log", "--oneline", "--graph", "-14")
spaces()

* 615c95e Revert "Merge pull request #3 from feature/shorter-summaries"
*   aa43145 Merge pull request #3 from feature/shorter-summaries
|\  
| * 002fc6f Shorter summaries
|/  
* 2cdc3d3 Revert "Merge pull request #2 from feature/show-gross-sales"
*   dbe8030 Merge pull request #2 from feature/show-gross-sales
|\  
| * 5506bdb Show gross sales in the example queries
|/  
*   796da12 Merge pull request #1 from feature/online-share
|\  
| * cb2cac8 Add a trusted example and a benchmark for online share by region
|/  
* 8cf3f20 FreshCart Genie space as code


,workspace,space title,catalog,examples,benchmarks,sample questions,edits (etag)
0,dev,[dev dana] FreshCart Sales & Stock Assistant,freshcart_dev,5,9,5,7
1,qa,FreshCart Sales & Stock Assistant [QA],freshcart_qa,5,9,5,6
2,prod,FreshCart Sales & Stock Assistant,freshcart,5,9,5,6


In [41]:
for ws in WS.values():
    ws.stop()
shutil.rmtree(LAB, ignore_errors=True)
print("lab cleaned up")

lab cleaned up


---
# 6 · Going live: one-time setup

### 6.1 · Databricks, in each workspace (dev, qa, prod)
| # | What | Why / how |
|---|---|---|
| 1 | **The FreshCart data** | deploy and run the data bundle (`../databricks`): it creates `freshcart_dev` / `freshcart_qa` / `freshcart` with the gold tables, metric views and the like-for-like function |
| 2 | **A serverless SQL warehouse** | named as in `databricks.yml` (`lookup: warehouse:`), or change the name there |
| 3 | **Group `freshcart-business-users`** | the people who may ask questions. They need `CAN_RUN` on the space (the bundle grants it), `USE CATALOG` / `USE SCHEMA` / `SELECT` on the FreshCart data (`governance/01_security.sql` in the data bundle), and access to the warehouse |
| 4 | **One service principal per environment** | the identity CI deploys and tests with. Give it workspace access, Databricks SQL access, `CAN_USE` on the warehouse, and `USE CATALOG` / `USE SCHEMA` / `SELECT` on that environment's catalog (the smoke test and the benchmarks run queries as it). Create an **OAuth secret** for it |
| 5 | **Pin the hosts** (recommended) | uncomment `host:` under each target in `databricks.yml`, so a target can never be deployed to the wrong workspace |

### 6.2 · GitHub
| # | What | Where |
|---|---|---|
| 1 | **A repository whose root is `genie-as-code-review/`** | copy the folder into a new repository (the workflow's paths assume that), default branch `main` |
| 2 | **Variables** `DEV_DATABRICKS_HOST`, `QA_DATABRICKS_HOST`, `PROD_DATABRICKS_HOST` | Settings → Secrets and variables → Actions → Variables |
| 3 | **Secrets** `DEV_SP_CLIENT_ID`, `DEV_SP_CLIENT_SECRET` (and `QA_…`, `PROD_…`) | Settings → Secrets and variables → Actions → Secrets (or as environment secrets of dev / qa / prod) |
| 4 | **Environments** `dev`, `qa`, `prod` | Settings → Environments. On `prod`: **required reviewers** and *deployment branches: `main` only*. Optionally reviewers on `qa` |
| 5 | **Branch protection on `main`** | require a pull request with an approval, and require the status check **Validate bundle** |

### 6.3 · The first deployment
1. **Is there already a curated dev space?** If yes, adopt it instead of creating a new one: `bundle generate
   genie-space --existing-id <id> --key freshcart_assistant --force`, fix the YAML, `bundle deployment bind`
   (the folder's README, *Approach B*, has the exact steps).
2. Push to `main` (or run the workflow with target `qa`) and watch dev and QA go green.
3. Read the benchmark gate's output: which questions are `BAD`, which `NEEDS_REVIEW`. Fix examples or benchmarks
   until you trust the result, and only then choose the threshold.
4. Run the workflow with target `prod` and approve.

---
# 7 · Runbook

| Situation | What you see | What to do |
|---|---|---|
| **Pull request is red** | `validate` failed: `json.tool` error with a line number, or `bundle validate` names an unknown field / unresolved variable / missing warehouse | fix the file on the branch and push again. If the JSON came from `bundle generate`, regenerate rather than hand-fix |
| **Smoke test failed** | `[FAIL] <question> (COMPLETED, 0 row(s))` or `(FAILED, ...)` | the space can't answer in that workspace: is the data there (data bundle run)? Does the service principal have `SELECT` and warehouse access? Was a table renamed? |
| **Benchmark gate failed** | `GATE FAILED: accuracy …` and the `❌` questions with a reason | a real regression: revert the merge (5.5). A benchmark whose expected SQL is out of date: fix the benchmark in a pull request. Genie wording variation on one question: improve the example or the instructions |
| **`GATE FAILED: no benchmark could be graded`** | every question `❔ NEEDS_REVIEW` | the space has no benchmarks, or none with an expected SQL; add benchmarks |
| **Prod waits** | the run shows *Waiting for review* | a reviewer of the `prod` environment approves it in the run's page |
| **Bad version in prod** | users complain | revert the merge, push, run with target `prod` (5.6) |
| **Someone edited prod in the UI** | `bundle plan -t prod` shows `1 to change` with no release pending | redo the edit in dev and release it, or let the next release overwrite it (5.7); then remove their `CAN_MANAGE` |
| **UI edits in dev were lost** | after a deployment to dev, the UI changes are gone | they were never exported: always `bundle generate … --force` and commit before anything deploys to dev. Use `--watch` while you work |
| **Two releases at once** | the second run shows *Waiting* | expected: `concurrency` queues runs |
| **Service principal secret expired** | `bundle validate` fails with an authentication error | create a new OAuth secret, update the GitHub secret |

---
# 8 · Simple bundle or full bundle?

This repository also contains the **full** version of the same space and process, [`genie_bundle/`](../genie_bundle/)
(explained in [`Genie_CICD_with_Declarative_Automation_Bundles.ipynb`](Genie_CICD_with_Declarative_Automation_Bundles.ipynb)).

| | Simple (`genie-as-code-review/`) | Full (`genie_bundle/`) |
|---|---|---|
| Space stored as | the JSON `bundle generate` writes | a bundle variable in YAML, with `${var.catalog}` |
| Catalog per environment | `set_catalog.sh` rewrites the dev catalog at deploy time | resolved by the bundle |
| Exporting UI edits from dev | `bundle generate --force` by hand | an hourly workflow opens a pull request with every UI change, tagged |
| Quality gates | validate · smoke test · benchmark gate (in GitHub Actions) | the same, plus SQL checks on every pull request, the gate as a **Databricks job**, a nightly prod run |
| Drift in qa/prod | overwritten silently by the next release | `check_drift.py` stops the release and backs up the live space |
| Rollback | `git revert` + pipeline | a rollback workflow to any tagged release |
| Environments | dev (personal), qa, prod | sandbox (personal), shared dev, qa, prod |
| Files to understand | 7 | about 15 |

**Start simple.** Move to the full bundle when several people edit the space every week (automatic exports), when
prod edits must never be lost unnoticed (drift gate), or when auditors ask for tagged releases.

---
# 9 · Summary and glossary

**The process in five sentences.** The Genie space is a JSON file in git. People improve it in the dev Genie UI and
export the result with `bundle generate`; the change goes through a pull request that is validated automatically.
Merging deploys it to dev (smoke test) and QA (benchmark gate) with `databricks bundle deploy`. A manual, approved run
promotes the same commit to prod, after the QA gate passed again in that run. Rolling back is reverting the commit and
running the same pipeline.

| Term | Meaning |
|---|---|
| **Bundle** (DAB) | a folder with `databricks.yml` that describes Databricks resources; deployed per target with the CLI |
| **Target** | one environment of a bundle (`dev`, `qa`, `prod`), with its own mode, variables and workspace |
| **Development / production mode** | per-user copies with a `[dev <name>]` prefix / exactly one copy in a fixed place |
| **Serialized space** | the whole content of a Genie space as one JSON document |
| **`bundle generate genie-space`** | exports a space from a workspace into the bundle (`--resource <key> --force` updates the existing file) |
| **`bundle deploy` / `bundle plan`** | make the workspace match the bundle / show what deploy would change |
| **`bundle deployment bind`** | make the bundle manage a space that already exists, instead of creating a new one |
| **Service principal** | a non-human identity; CI deploys and tests as one, per environment |
| **GitHub environment** | a named deployment target in GitHub with its own secrets and optional required reviewers |
| **Smoke test** | a quick check that the deployed space answers at all |
| **Benchmark / benchmark gate** | a question with its expected SQL / the check that enough benchmarks are answered correctly |
| **Drift** | a difference between what is deployed and what is in git, caused by an edit outside the pipeline |
| **Revert** | a new commit that undoes an earlier one; the safe way to roll back |